# NB-14 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-14**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-14.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This applies the operator and coordinate discussion to retained JAO zonal PTDF and constraint records. Zonal geometry depends on the interval, stage, revision, carrier, and sign convention.

- [Volume V, Chapter 4 — Operators, coordinates, and representations](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c4)
- [Volume II, Chapter 8 — From prices back to the graph](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c8)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-14', require_key=True)
print(setup_status)


# NB-14 — JAO active constraints to observed zonal geometry
Companion to Volume II §8, Volume IV §21, and Volume V §4.
Prerequisite: NB-15's distinction between magnitude and direction. We use a
retained, sealed EWB export of actual JAO observations, not controlled substitutes.
The notebook reads retained observations locally and calls the authenticated private methods API; it creates no synthetic data.

1. Audit interval, stage, revision, and carrier identity.
2. Join observed PTDFs to compatible active constraints.
3. Compare balanced-transfer geometry under two declared metrics.
4. Align changing active-row support without inventing missing sensitivities.

In [ ]:
NOTEBOOK_ID = 'NB-14'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

carriers,audit=inputs['carriers'],inputs['audit']
zones,paths=inputs['zones'],inputs['paths']
print('Local raw PTDF inspection:',inputs['raw_bundle']['queries']['ptdf']['response']['data'][:1])
audited=execute('audit','zonal_operator_audit',inputs['requests']['audit'])
assert audited['support_ledger']['excluded_hubs']==audit['excluded_hubs']
matrices=[]
for j,(window,meta) in enumerate(zip(audited['audited_zonal_operator']['windows'],inputs['windows'],strict=True)):
    assert window['constraint_ids']==meta['rows'] and window['start']==meta['time'] and window['node_ids']==zones
    e=np.asarray([t['physical_source_to_sink'] for t in window['transfers']]).T
    assert [t['path_id'] for t in window['transfers']]==[s+'->'+t for s,t in paths]
    np.testing.assert_allclose(np.asarray([t['payout_sink_minus_source'] for t in window['transfers']]).T,-e)
    m={**meta,'psi':np.asarray(window['values']),'exposure':e,'h':np.asarray(meta['h']),'duals':np.asarray(meta['duals'])}
    for metric in ('identity','severity'):
        metric_input=dict(kind='identity') if metric=='identity' else dict(kind='diagonal',constraint_ids=meta['rows'],values=abs(m['duals']).tolist())
        request=dict(inputs=dict(exposure=dict(constraint_ids=meta['rows'],path_ids=[s+'->'+t for s,t in paths],values=e.tolist()),metric=metric_input),
            parameters=dict(outputs=['magnitude','cosine','gram']),
            context=dict(input_origin='user_supplied',exposure_kind='structural_exposure',
                exposure_convention='JAO active zonal source injection +1, sink withdrawal -1; no nodal recovery',
                exposure_unit='MW/MW',metric_unit='dimensionless' if metric=='identity' else 'EUR/MWh',
                model_context='NB14-PT15M-v1 '+meta['time']+' latest descriptive sample'))
        geo=execute(f'window{j}_{metric}','exposure_geometry',request,['audit'])
        m[metric]=SimpleNamespace(gram=np.asarray(geo['gram']),magnitude=np.asarray(geo['magnitude']),
            cosine=np.asarray(geo['cosine'],dtype=float),valid=np.asarray(geo['valid_path_mask'],dtype=bool))
    matrices.append(m)
print(json.dumps(audit,indent=2))
for label,item in carriers['selected'].items():print(label,item['name'],item['eic'],item['type'])

## Contract audit before the matrix
The export contains all returned pages for two starts, 00:00 and 00:15 UTC on
13 October 2026. Each record lasts **15 minutes**, despite PT60M in both dataset
descriptors. `NB14-PT15M-v1` explicitly overrides that descriptor for this sealed
sample only; it does not change the upstream contract or infer other dates.
The half-hour delivery span contains two observations, not two hourly products.

The join uses start, end, compound CNEC key and payload hash, then verifies source
publication, collector receipt and parser version. Multiple revisions are rejected
rather than averaged or silently picked. This exact payload equality is supported
for these two active datasets; it is not a universal cross-endpoint join rule.

JAO describes active FB constraints as **after market coupling** in its
[publication update](https://www.jao.eu/news/core-da-publication-tool-update-planned).
This export's publication also occurs after these delivery intervals. Neither
latest view nor collector receipt establishes auction-time information.
The query covers active FB rows, not a complete grid or the separate active-LTA
contribution. No full-price reconstruction or tradable capacity claim follows.

In [ ]:
for m in matrices:
    assert all(x>0 for x in m['publication_lag_hours'])
    print(m['time'],{'active_rows':len(m['rows']),
        'hours_after_interval_end_to_publication':round(min(m['publication_lag_hours']),3),
        'matrix_shape':m['psi'].shape})
    for i,row in enumerate(m['rows']): print(' C'+str(i+1),row)

## Carriers and sign convention
AT, CZ and SK are ordinary bidding-zone entries verified against EWB's
EIC-backed zone registry; its selected entries and source checksum are retained
in the carrier receipt. JAO's [Core handbook](https://www.jao.eu/sites/default/files/2022-06/CORE%20-%20Publication%20Tool%20for%20DA%20CCM%20Publication%20Handbook%201.7.pdf)
documents separate hub coordinates, including virtual ALEGrO hubs. We retain
all 23 returned hub labels in the raw export, but use only AT/CZ/SK here.
No ALBE→BE or other virtual-hub mapping is inferred from string similarity.
Constraint `hub_from/hub_to` fields identify the maximum-sensitivity exchange,
not the physical branch endpoints or the ordering of all PTDF columns.

A commercial source→sink transfer injects +1 at source and withdraws −1 at sink:
**E = Ψ(e_source − e_sink)**, in MW constraint-flow response per MW transfer.
NB-15 used a payout vector with the opposite sign. We explicitly distinguish the
two conventions; balanced differences remove the common slack contribution.
Reversal negates exposure, while CZ→SK + AT→CZ = AT→SK.

In [ ]:
labels=[source+'→'+sink for source,sink in paths]
for m in matrices:
    e=m['exposure'];h=m['h']
    np.testing.assert_allclose(h.sum(axis=0),0)
    np.testing.assert_allclose(e[:,0],-e[:,1],atol=1e-12)
    np.testing.assert_allclose(e[:,0]+e[:,2],e[:,3],atol=1e-12)
    # Centering each observed row changes its reference, not balanced differences.
    np.testing.assert_allclose((m['psi']-m['psi'].mean(axis=1,keepdims=True))@h,e,atol=1e-12)
from matplotlib.colors import LinearSegmentedColormap
cmap=LinearSegmentedColormap.from_list('nord_diverging',['#5E81AC','#ECEFF4','#BF616A'])
first=matrices[0]
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,array,title,ticks in zip(axes,[first['psi'],first['exposure']],
        ['Observed zone-to-slack PTDF','Balanced commercial-transfer exposure'],[list(zones),labels]):
    bound=abs(array).max();im=ax.imshow(array,cmap=cmap,vmin=-bound,vmax=bound,aspect='auto')
    ax.set_xticks(range(len(ticks)),ticks);ax.set_yticks(range(len(first['rows'])),['C'+str(i+1) for i in range(len(first['rows']))])
    ax.set(title=title,ylabel='Active CNEC row · 00:00 UTC');fig.colorbar(im,ax=ax,label='MW/MW')
fig.tight_layout();plt.show()

## Equal weights and contemporaneous economic severity
Identity weights describe the returned active-row geometry. The second metric
is diag(abs(published shadow price)), with units EUR/MWh. It weights economically
severe rows in this observed interval; it is not covariance, a forecast, or an
estimate of the missing full system. Square-root weights form the feature rows.
A negative published dual would retain its sign in raw data; this declared PSD
metric uses its absolute value. Norms have different units across metrics.
Zero-norm paths have undefined cosine, represented by NaN, not zero similarity.

In [ ]:
for m in matrices:
    e=m['exposure'];severity=m['severity'];factor=np.sqrt(abs(m['duals']))[:,None]*e
    np.testing.assert_allclose(factor.T@factor,severity.gram,atol=1e-10)
    for metric in ('identity','severity'):
        geo=m[metric]
        assert np.linalg.eigvalsh(geo.gram).min()>-1e-9
        np.testing.assert_allclose(geo.magnitude[0],geo.magnitude[1],atol=1e-12)
        if geo.valid[0]: np.testing.assert_allclose(geo.cosine[0,1],-1,atol=1e-12)
        print(m['time'],metric,{'norms':geo.magnitude.round(6).tolist(),'zero_mask':(~geo.valid).tolist()})
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,metric in zip(axes[:2],('identity','severity')):
    geo=first[metric];im=ax.imshow(geo.cosine,vmin=-1,vmax=1,cmap=cmap)
    ax.set_xticks(range(4),labels,rotation=35,ha='right');ax.set_yticks(range(4),labels)
    ax.set_title(metric+' · signed cosine');fig.colorbar(im,ax=ax)
axes[2].bar(range(4),first['severity'].magnitude,color='#88C0D0')
axes[2].set_xticks(range(4),labels,rotation=35,ha='right')
axes[2].set(title='Severity-weighted magnitude',ylabel='sqrt(EUR/MWh) × MW/MW')
fig.tight_layout();plt.show()

## Support changes are not zero sensitivities
The second MTU has one additional published active CNEC. This is an active-set
observation, not evidence of a physical line addition or outage. Retain the union
with an explicit absence mask. Compare operators only on the common CNEC support
and the same zone coordinates; report newly observed rows separately.
Missing active rows cannot be filled with zero PTDF. Even equal matrix dimensions
would not establish equal row identity. Two intervals cannot establish stability.

In [ ]:
first,second=matrices
common=sorted(set(first['rows'])&set(second['rows']));union=sorted(set(first['rows'])|set(second['rows']))
panel=np.full((2,len(union),len(paths)),np.nan)
for i,m in enumerate(matrices):
    for j,row in enumerate(m['rows']):panel[i,union.index(row)]=m['exposure'][j]
assert common
e0=first['exposure'][[first['rows'].index(k) for k in common]]
e1=second['exposure'][[second['rows'].index(k) for k in common]]
change=dict(common=common,union=union,added=sorted(set(second['rows'])-set(first['rows'])),
    removed=sorted(set(first['rows'])-set(second['rows'])),panel=panel,
    common_operator_delta_norm=float(np.linalg.norm(e1-e0)),
    common_gram_delta_norm=float(np.linalg.norm(e1.T@e1-e0.T@e0)),unmatched_coordinate_count=int(np.isnan(panel).sum()))

print({k:v for k,v in change.items() if k!='panel'})
assert change['unmatched_coordinate_count']>0
assert np.isnan(change['panel'][0]).any()
fig,axes=plt.subplots(1,2,figsize=(12,4))
cmap=cmap.copy();cmap.set_bad('#4C566A');bound=np.nanmax(abs(change['panel']))
for i,ax in enumerate(axes):
    im=ax.imshow(np.ma.masked_invalid(change['panel'][i]),aspect='auto',cmap=cmap,vmin=-bound,vmax=bound)
    ax.set_xticks(range(4),labels);ax.set_yticks(range(len(change['union'])),['U'+str(j+1) for j in range(len(change['union']))])
    ax.set(title=('00:00' if i==0 else '00:15')+' UTC · grey = absent/unknown',ylabel='Union CNEC row')
    fig.colorbar(im,ax=ax,label='MW/MW')
fig.tight_layout();plt.show()
for i,row in enumerate(change['union']): print('U'+str(i+1),row)

## Exercise and next boundary
Does treating the grey cell as zero mean a previously inactive constraint had
zero sensitivity? No: publication support says whether a row is present, not
its unobserved coefficients. The answer scaffold counts observed rows and shows
the common-support delta separately.

Extending this example requires another complete, revision-compatible observed
export and validated carriers. Auction-time prediction needs an admissible earlier
publication. These JAO zonal sensitivities cannot be transferred to PJM nodes or
treated as an electrical reconstruction of the European network.

In [ ]:
print('Observed rows:',[len(m['rows']) for m in matrices],'; common rows:',len(change['common']))
print('Common-support operator delta:',change['common_operator_delta_norm'])

In [ ]:
bad=copy.deepcopy(inputs['requests']['audit']);bad['inputs']['constraint_domain_records'].pop()
reject('zonal_operator_audit',bad)
bad=copy.deepcopy(next(item['request'] for item in execution_plan if item['name']=='window0_severity'))
bad['inputs']['metric']['constraint_ids']=list(reversed(bad['inputs']['metric']['constraint_ids']))
reject('exposure_geometry',bad)
support={k:v for k,v in change.items() if k!='panel'}
support['absent_mask']=np.isnan(change['panel']).tolist()
support['values']=np.where(np.isnan(change['panel']),None,change['panel']).tolist()
operators=[]
for m in matrices:
    row={k:m[k].tolist() if isinstance(m[k],np.ndarray) else m[k] for k in ('time','rows','psi','exposure','duals')}
    for metric in ('identity','severity'):row[metric]={k:getattr(m[metric],k).tolist() for k in ('gram','magnitude','cosine','valid')}
    operators.append(row)
(session.output/'consumer-summary.json').write_text(json.dumps(dict(audit=audit,carriers=carriers,support_change=support,operators=operators),indent=2,allow_nan=False)+'\n')
summary=session.finish({'complete_pages':True,'compatible_payload_join':True,'PT15M_explicit_override':True,
    'carrier_identity':True,'balanced_sign_and_composition':True,'PSD_metrics':True,'missing_rows_not_zero':True,'new_synthetic_data':False},
    {'scope':'Two observed active-set MTUs; latest descriptive only','snapshot':audit['snapshot']})